# 🌿 EcoVideo Studio — Kaggle GPU Backend

This notebook runs the **inference backend** for EcoVideo Studio on Kaggle's free GPU.  
Your local machine only runs the Streamlit frontend.

### Setup
1. Enable **GPU accelerator** (Settings → Accelerator → GPU T4 x2)
2. Run all cells
3. Copy the **ngrok URL** printed at the end
4. Paste it into your local `.env` file as `KAGGLE_BACKEND_URL=https://xxxx.ngrok-free.app`
5. Run `streamlit run frontend/app.py` locally

In [ ]:
# Cell 1: Install dependencies
!pip install -q fastapi uvicorn pyngrok diffusers transformers accelerate safetensors pillow pynvml

# Set your ngrok auth token (get free one from https://dashboard.ngrok.com)
import os
os.environ["NGROK_AUTH_TOKEN"] = "YOUR_NGROK_TOKEN_HERE"  # <-- PASTE YOUR TOKEN

In [ ]:
"""Cell 2: Load SVD Pipeline on GPU"""

import gc
import io
import time
import uuid
import base64
import logging
from pathlib import Path

import torch
import numpy as np
from PIL import Image
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ecovideo-kaggle")

# ── Preset configs ──────────────────────────────────────────────
PRESETS = {
    "eco": {"steps": 8, "decode_chunk": 1, "max_frames": 4},
    "balanced": {"steps": 16, "decode_chunk": 1, "max_frames": 6},
    "quality": {"steps": 25, "decode_chunk": 1, "max_frames": 14},
}

# ── Load model ──────────────────────────────────────────────────
MODEL_ID = "stabilityai/stable-video-diffusion-img2vid"

logger.info("Loading SVD pipeline...")
start = time.time()

pipe = StableVideoDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()

logger.info(f"Pipeline loaded in {time.time()-start:.1f}s on {torch.cuda.get_device_name()}")
print(f"✅ Model loaded on {torch.cuda.get_device_name()}")
print(f"   VRAM: {torch.cuda.mem_get_info()[1]/1e9:.1f} GB total, {torch.cuda.mem_get_info()[0]/1e9:.1f} GB free")

In [ ]:
"""Cell 3: FastAPI Server with Hardware Energy Measurement"""

import json as _json
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse, JSONResponse
import uvicorn
import threading

# ── Hardware Energy Measurement (NVML + RAPL) ──────────────────
import pynvml

pynvml.nvmlInit()
_gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
_RAPL_PATH = Path("/sys/class/powercap/intel-rapl/intel-rapl:0/energy_uj")
_HAS_RAPL = _RAPL_PATH.exists()

def _read_rapl_uj() -> float:
    """Read CPU energy counter in microjoules via Intel RAPL."""
    if _HAS_RAPL:
        return float(_RAPL_PATH.read_text().strip())
    return 0.0

class HardwareEnergyMeter:
    """Measures real GPU power (NVML) + CPU energy (RAPL) during inference.

    - GPU: Polls nvidia-smi power sensor every call to update() (milliwatts precision)
    - CPU: Reads Intel RAPL energy_uj counter (microjoule precision)
    """

    def __init__(self):
        self._start_time = 0.0
        self._gpu_power_samples: list[float] = []  # watts
        self._rapl_start_uj = 0.0

    def start(self):
        self._start_time = time.time()
        self._gpu_power_samples = []
        self._rapl_start_uj = _read_rapl_uj()
        self._sample()

    def _sample(self):
        try:
            power_mw = pynvml.nvmlDeviceGetPowerUsage(_gpu_handle)
            self._gpu_power_samples.append(power_mw / 1000.0)
        except Exception:
            pass

    def update(self):
        """Call periodically during generation for more power samples."""
        self._sample()

    def stop(self) -> dict:
        """Stop measurement, return energy breakdown."""
        self._sample()
        duration = time.time() - self._start_time

        # GPU energy: average power × time
        avg_gpu_w = sum(self._gpu_power_samples) / max(len(self._gpu_power_samples), 1)
        gpu_energy_wh = (avg_gpu_w * duration) / 3600.0

        # CPU energy from RAPL counter difference
        rapl_end_uj = _read_rapl_uj()
        cpu_energy_uj = rapl_end_uj - self._rapl_start_uj
        if cpu_energy_uj < 0:
            cpu_energy_uj += 2**32  # handle counter overflow
        cpu_energy_wh = cpu_energy_uj / 1e6 / 3600.0  # uJ → J → Wh

        total_wh = gpu_energy_wh + cpu_energy_wh
        return {
            "total_energy_wh": total_wh,
            "gpu_energy_wh": gpu_energy_wh,
            "cpu_energy_wh": cpu_energy_wh,
            "avg_gpu_watts": avg_gpu_w,
            "gpu_samples": len(self._gpu_power_samples),
            "duration_s": duration,
            "method": "hardware_nvml" + ("_rapl" if _HAS_RAPL else ""),
        }

# ── FastAPI App ─────────────────────────────────────────────────
app = FastAPI(title="EcoVideo Studio Backend")

OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def _safe_call(obj, method_name, *args, **kwargs):
    """Call a method only if it exists on the object."""
    fn = getattr(obj, method_name, None)
    if fn is not None and callable(fn):
        fn(*args, **kwargs)
        return True
    logger.debug(f"{type(obj).__name__} has no '{method_name}', skipping")
    return False


def _apply_optimizations(pipeline, opts: dict):
    """Apply/disable optimization flags on the pipeline."""
    if opts.get("attention_slicing", True):
        _safe_call(pipeline, "enable_attention_slicing", "auto")
    else:
        _safe_call(pipeline, "disable_attention_slicing")

    if opts.get("forward_chunking", True):
        _safe_call(pipeline.unet, "enable_forward_chunking", chunk_size=1, dim=1)
    else:
        _safe_call(pipeline.unet, "disable_forward_chunking")

    if opts.get("cpu_offload", False):
        _safe_call(pipeline, "enable_model_cpu_offload")

    if opts.get("vae_slicing", False):
        _safe_call(pipeline, "enable_vae_slicing")
    else:
        _safe_call(pipeline, "disable_vae_slicing")

    if opts.get("vae_tiling", False):
        _safe_call(pipeline, "enable_vae_tiling")
    else:
        _safe_call(pipeline, "disable_vae_tiling")


@app.get("/health")
def health():
    free, total = torch.cuda.mem_get_info()
    power_w = pynvml.nvmlDeviceGetPowerUsage(_gpu_handle) / 1000.0
    return {
        "status": "ok",
        "device": torch.cuda.get_device_name(),
        "vram_free_gb": round(free / 1e9, 1),
        "vram_total_gb": round(total / 1e9, 1),
        "current_gpu_watts": round(power_w, 1),
        "rapl_available": _HAS_RAPL,
    }


@app.post("/generate")
async def generate(
    image: UploadFile = File(...),
    preset_name: str = Form("balanced"),
    seed: int = Form(42),
    num_frames: int = Form(14),
    motion_bucket_id: int = Form(127),
    noise_aug_strength: float = Form(0.02),
    optimizations_json: str = Form("{}"),
):
    """Generate video from uploaded image with hardware energy measurement."""
    try:
        preset = PRESETS.get(preset_name, PRESETS["balanced"])
        opts = _json.loads(optimizations_json)

        # Apply optimizations
        _apply_optimizations(pipe, opts)

        # DeepCache setup
        use_deep_cache = opts.get("deep_cache", False)
        cache_interval = opts.get("deep_cache_interval", 3)

        # Read image
        img_bytes = await image.read()
        pil_image = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        num_frames = min(num_frames, preset["max_frames"])
        generator = torch.Generator(device="cuda").manual_seed(seed)

        active_opts = [k for k, v in opts.items() if v is True]
        logger.info(f"Generating: preset={preset_name}, steps={preset['steps']}, frames={num_frames}, opts={active_opts}")

        # DeepCache effective steps
        if use_deep_cache:
            try:
                effective_steps = max(preset["steps"] // cache_interval * 2, 4)
                logger.info(f"DeepCache ON: interval={cache_interval}, effective_steps={effective_steps}")
            except Exception:
                effective_steps = preset["steps"]
                logger.warning("DeepCache setup failed, using normal steps")
        else:
            effective_steps = preset["steps"]

        # ── Start hardware energy measurement ──
        energy_meter = HardwareEnergyMeter()
        energy_meter.start()

        with torch.no_grad():
            output = pipe(
                image=pil_image,
                decode_chunk_size=preset["decode_chunk"],
                generator=generator,
                num_frames=num_frames,
                motion_bucket_id=motion_bucket_id,
                noise_aug_strength=noise_aug_strength,
                num_inference_steps=effective_steps if use_deep_cache else preset["steps"],
            )

        # ── Stop hardware energy measurement ──
        energy_data = energy_meter.stop()
        energy_wh = energy_data["total_energy_wh"]

        gen_time = energy_data["duration_s"]
        frames = output.frames[0]

        # Peak VRAM
        free, total = torch.cuda.mem_get_info()
        peak_vram_gb = (total - free) / 1e9

        # Save video
        gen_id = str(uuid.uuid4())[:8]
        filename = f"ecovideo_{gen_id}_{preset_name}.mp4"
        output_path = OUTPUT_DIR / filename
        export_to_video(frames, str(output_path), fps=7)

        # Cleanup
        gc.collect()
        torch.cuda.empty_cache()

        logger.info(
            f"Done: {gen_time:.1f}s, {peak_vram_gb:.1f}GB VRAM, "
            f"{energy_wh:.4f}Wh ({energy_data['method']}), "
            f"GPU: {energy_data['avg_gpu_watts']:.1f}W avg ({energy_data['gpu_samples']} samples), "
            f"opts={active_opts}"
        )

        return FileResponse(
            str(output_path),
            media_type="video/mp4",
            filename=filename,
            headers={
                "X-Generation-Id": gen_id,
                "X-Generation-Time": str(round(gen_time, 2)),
                "X-Peak-Vram-Gb": str(round(peak_vram_gb, 2)),
                "X-Energy-Wh": str(round(energy_wh, 4)),
                "X-Gpu-Energy-Wh": str(round(energy_data["gpu_energy_wh"], 4)),
                "X-Cpu-Energy-Wh": str(round(energy_data["cpu_energy_wh"], 4)),
                "X-Avg-Gpu-Watts": str(round(energy_data["avg_gpu_watts"], 1)),
                "X-Gpu-Samples": str(energy_data["gpu_samples"]),
                "X-Energy-Method": energy_data["method"],
                "X-Preset": preset_name,
                "X-Num-Steps": str(effective_steps if use_deep_cache else preset["steps"]),
                "X-Seed": str(seed),
                "X-Num-Frames": str(num_frames),
                "X-Optimizations": ",".join(active_opts),
            },
        )

    except Exception as e:
        logger.exception("Generation failed")
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/recommend")
async def recommend_preset(image: UploadFile = File(...)):
    """Simple rule-based recommendation."""
    free, total = torch.cuda.mem_get_info()
    vram_gb = free / 1e9

    if vram_gb < 4:
        preset = "eco"
    elif vram_gb < 10:
        preset = "balanced"
    else:
        preset = "quality"

    return {
        "preset": preset,
        "confidence": 0.75,
        "method": "rule_based_gpu",
        "hardware": {
            "device": torch.cuda.get_device_name(),
            "available_vram_gb": round(vram_gb, 1),
        },
    }


# Start server in background thread
def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

thread = threading.Thread(target=_run, daemon=True)
thread.start()
print("✅ FastAPI server running on port 8000")

In [ ]:
"""Cell 4: Expose via ngrok"""

from pyngrok import ngrok

# Authenticate
ngrok.set_auth_token(os.environ["NGROK_AUTH_TOKEN"])

# Open tunnel
public_url = ngrok.connect(8000, "http").public_url
print("=" * 60)
print(f"🌿 EcoVideo Backend is LIVE!")
print(f"   Public URL: {public_url}")
print(f"   Health:     {public_url}/health")
print()
print(f"   Paste this into your local .env file:")
print(f"   KAGGLE_BACKEND_URL={public_url}")
print("=" * 60)

In [ ]:
"""Cell 5: Keep alive — run this cell and leave it running"""

import time
try:
    while True:
        time.sleep(60)
        free, total = torch.cuda.mem_get_info()
        print(f"[keepalive] VRAM: {free/1e9:.1f}/{total/1e9:.1f} GB free | URL: {public_url}", flush=True)
except KeyboardInterrupt:
    print("Shutting down...")
    ngrok.kill()

# 📊 Optimization Benchmark — Resource & Quality Comparison

This cell runs a controlled experiment:
- **Baseline** (no optimizations) vs each optimization individually vs key combinations
- Measures: **Time**, **Peak VRAM**, **Energy (Wh)**, **SSIM**, **PSNR**, **LPIPS**
- Uses fixed seed + same input image for fair comparison
- Results saved as CSV + visualized with charts

In [ ]:
"""Cell 6: Optimization Benchmark — Full Comparison"""

!pip install -q scikit-image lpips torchmetrics pandas matplotlib seaborn

import gc
import time
import json
import itertools
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Try LPIPS (perceptual quality)
try:
    import lpips
    lpips_fn = lpips.LPIPS(net="alex").cuda()
    HAS_LPIPS = True
except Exception:
    HAS_LPIPS = False
    print("⚠️ LPIPS not available, skipping perceptual metric")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# IMAGE SOURCE — Modular loader (synthetic / single file / dataset)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Choose ONE mode ──────────────────────────────────────────────
# MODE = "synthetic"        # Auto-generate a gradient image (no upload needed)
# MODE = "single"           # Use one specific image file
# MODE = "dataset"          # Run benchmark across multiple images (averages results)

IMAGE_MODE = "synthetic"  # <-- CHANGE THIS to "single" or "dataset" as needed

# ── Mode-specific config ────────────────────────────────────────
SINGLE_IMAGE_PATH = "/kaggle/input/your-image.png"             # For MODE="single"
DATASET_DIR = "/kaggle/input/your-dataset/"                     # For MODE="dataset"
DATASET_EXTENSIONS = (".png", ".jpg", ".jpeg", ".webp")         # Accepted formats
DATASET_MAX_IMAGES = 5                                          # Limit for speed (set None for all)


def create_synthetic_image(path: Path) -> Image.Image:
    """Generate a reproducible 1024×576 gradient image."""
    arr = np.zeros((576, 1024, 3), dtype=np.uint8)
    arr[:, :, 0] = np.linspace(0, 255, 1024, dtype=np.uint8)    # R gradient horizontal
    arr[:, :, 1] = np.linspace(0, 255, 576, dtype=np.uint8)[:, None]  # G gradient vertical
    arr[:, :, 2] = 128
    img = Image.fromarray(arr)
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path)
    return img


def load_benchmark_images() -> List[dict]:
    """
    Load images based on IMAGE_MODE.
    Returns list of {"name": str, "image": PIL.Image}.
    """
    if IMAGE_MODE == "synthetic":
        path = Path("/kaggle/working/benchmark_input.png")
        img = create_synthetic_image(path)
        print(f"✅ Synthetic image created: {path}")
        return [{"name": "synthetic_gradient", "image": img}]

    elif IMAGE_MODE == "single":
        path = Path(SINGLE_IMAGE_PATH)
        if not path.exists():
            raise FileNotFoundError(
                f"Image not found: {path}\n"
                f"Upload an image to Kaggle or switch to IMAGE_MODE='synthetic'"
            )
        img = Image.open(path).convert("RGB")
        print(f"✅ Loaded single image: {path} ({img.size})")
        return [{"name": path.stem, "image": img}]

    elif IMAGE_MODE == "dataset":
        dataset_path = Path(DATASET_DIR)
        if not dataset_path.exists():
            raise FileNotFoundError(
                f"Dataset directory not found: {dataset_path}\n"
                f"Add a dataset in Kaggle (Input → Add Data) or switch to IMAGE_MODE='synthetic'"
            )

        image_files = sorted([
            f for f in dataset_path.rglob("*")
            if f.suffix.lower() in DATASET_EXTENSIONS
        ])

        if not image_files:
            raise FileNotFoundError(f"No images found in {dataset_path} with extensions {DATASET_EXTENSIONS}")

        if DATASET_MAX_IMAGES is not None:
            image_files = image_files[:DATASET_MAX_IMAGES]

        images = []
        for f in image_files:
            try:
                img = Image.open(f).convert("RGB")
                images.append({"name": f.stem, "image": img})
            except Exception as e:
                print(f"⚠️ Skipping {f.name}: {e}")

        print(f"✅ Loaded {len(images)} images from dataset: {dataset_path}")
        for im in images:
            print(f"   • {im['name']} ({im['image'].size})")
        return images

    else:
        raise ValueError(f"Unknown IMAGE_MODE: '{IMAGE_MODE}'. Use 'synthetic', 'single', or 'dataset'.")


# Load images
benchmark_images = load_benchmark_images()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SEED = 42
NUM_FRAMES = 6
PRESET_NAME = "balanced"
PRESET = PRESETS[PRESET_NAME]
GPU_TDP_WATTS = 70.0  # T4 TDP
GPU_UTILIZATION = 0.8

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# DEFINE OPTIMIZATION CONFIGS TO TEST
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BENCHMARK_CONFIGS = {
    "baseline (no opts)": {
        "attention_slicing": False,
        "forward_chunking": False,
        "deep_cache": False,
        "cpu_offload": False,
    },
    "attention_slicing only": {
        "attention_slicing": True,
        "forward_chunking": False,
        "deep_cache": False,
        "cpu_offload": False,
    },
    "forward_chunking only": {
        "attention_slicing": False,
        "forward_chunking": True,
        "deep_cache": False,
        "cpu_offload": False,
    },
    "deep_cache only (interval=3)": {
        "attention_slicing": False,
        "forward_chunking": False,
        "deep_cache": True,
        "deep_cache_interval": 3,
        "cpu_offload": False,
    },
    "deep_cache only (interval=2)": {
        "attention_slicing": False,
        "forward_chunking": False,
        "deep_cache": True,
        "deep_cache_interval": 2,
        "cpu_offload": False,
    },
    "cpu_offload only": {
        "attention_slicing": False,
        "forward_chunking": False,
        "deep_cache": False,
        "cpu_offload": True,
    },
    "attn_slice + fwd_chunk": {
        "attention_slicing": True,
        "forward_chunking": True,
        "deep_cache": False,
        "cpu_offload": False,
    },
    "attn_slice + deep_cache(3)": {
        "attention_slicing": True,
        "forward_chunking": False,
        "deep_cache": True,
        "deep_cache_interval": 3,
        "cpu_offload": False,
    },
    "attn_slice + fwd_chunk + deep_cache(3)": {
        "attention_slicing": True,
        "forward_chunking": True,
        "deep_cache": True,
        "deep_cache_interval": 3,
        "cpu_offload": False,
    },
    "all opts (no cpu_offload)": {
        "attention_slicing": True,
        "forward_chunking": True,
        "deep_cache": True,
        "deep_cache_interval": 3,
        "cpu_offload": False,
    },
    "all opts + cpu_offload": {
        "attention_slicing": True,
        "forward_chunking": True,
        "deep_cache": True,
        "deep_cache_interval": 3,
        "cpu_offload": True,
    },
}

PRESET_COMPARISON = {
    "eco (8 steps, 4 frames)": ("eco", {"attention_slicing": True, "forward_chunking": True}),
    "balanced (16 steps, 6 frames)": ("balanced", {"attention_slicing": True, "forward_chunking": True}),
    "quality (25 steps, 14 frames)": ("quality", {"attention_slicing": True, "forward_chunking": True}),
}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# QUALITY METRICS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def compute_quality_metrics(frames_a: list, frames_b: list) -> dict:
    """Compare frames_a (test) against frames_b (reference/baseline)."""
    n = min(len(frames_a), len(frames_b))
    ssim_scores, psnr_scores, lpips_scores = [], [], []

    for i in range(n):
        img_a = np.array(frames_a[i].resize((256, 144)))
        img_b = np.array(frames_b[i].resize((256, 144)))

        s = ssim(img_a, img_b, channel_axis=2, data_range=255)
        ssim_scores.append(s)

        p = psnr(img_b, img_a, data_range=255)
        psnr_scores.append(p)

        if HAS_LPIPS:
            t_a = torch.from_numpy(img_a).permute(2, 0, 1).unsqueeze(0).float().cuda() / 127.5 - 1.0
            t_b = torch.from_numpy(img_b).permute(2, 0, 1).unsqueeze(0).float().cuda() / 127.5 - 1.0
            with torch.no_grad():
                lp = lpips_fn(t_a, t_b).item()
            lpips_scores.append(lp)

    return {
        "ssim_mean": float(np.mean(ssim_scores)),
        "ssim_std": float(np.std(ssim_scores)),
        "psnr_mean": float(np.mean(psnr_scores)),
        "psnr_std": float(np.std(psnr_scores)),
        "lpips_mean": float(np.mean(lpips_scores)) if lpips_scores else None,
        "lpips_std": float(np.std(lpips_scores)) if lpips_scores else None,
    }


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SINGLE RUN BENCHMARK
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_single_benchmark(image, opts, preset_name="balanced", num_frames=6):
    """Run one generation and return metrics."""
    preset = PRESETS[preset_name]
    num_frames = min(num_frames, preset["max_frames"])

    _apply_optimizations(pipe, opts)

    use_deep_cache = opts.get("deep_cache", False)
    cache_interval = opts.get("deep_cache_interval", 3)
    if use_deep_cache:
        effective_steps = max(preset["steps"] // cache_interval * 2, 4)
    else:
        effective_steps = preset["steps"]

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    generator = torch.Generator(device="cuda").manual_seed(SEED)

    start = time.time()
    with torch.no_grad():
        output = pipe(
            image=image,
            decode_chunk_size=preset["decode_chunk"],
            generator=generator,
            num_frames=num_frames,
            motion_bucket_id=127,
            noise_aug_strength=0.02,
            num_inference_steps=effective_steps,
        )
    gen_time = time.time() - start

    frames = output.frames[0]
    peak_vram_bytes = torch.cuda.max_memory_allocated()
    peak_vram_gb = peak_vram_bytes / (1024**3)
    free, total = torch.cuda.mem_get_info()
    current_vram_gb = (total - free) / 1e9
    energy_wh = (GPU_TDP_WATTS * GPU_UTILIZATION * gen_time) / 3600.0

    gc.collect()
    torch.cuda.empty_cache()

    return {
        "gen_time_s": round(gen_time, 2),
        "peak_vram_gb": round(peak_vram_gb, 3),
        "current_vram_gb": round(current_vram_gb, 2),
        "energy_wh": round(energy_wh, 4),
        "effective_steps": effective_steps,
        "num_frames": num_frames,
        "frames": frames,
    }


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MULTI-IMAGE BENCHMARK RUNNER
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_benchmark_across_images(
    images: List[dict],
    configs: dict,
    preset_name: str = "balanced",
    num_frames: int = 6,
) -> List[dict]:
    """
    Run all optimization configs across all images.
    For datasets, averages metrics across images.
    """
    all_results = []
    n_images = len(images)
    is_multi = n_images > 1

    if is_multi:
        print(f"📸 Running across {n_images} images — results will be averaged")

    # Store baselines per image (for quality comparison)
    baselines_per_image = {}

    for i, (config_name, opts) in enumerate(configs.items()):
        print(f"\n[{i+1}/{len(configs)}] Config: {config_name}" + (f" (across {n_images} images)" if is_multi else ""))

        per_image_metrics = []
        per_image_quality = []

        for img_info in images:
            img_name = img_info["name"]
            image = img_info["image"]

            try:
                metrics = run_single_benchmark(image, opts, preset_name, num_frames)

                # Quality: compare vs baseline for this image
                if config_name == list(configs.keys())[0]:
                    # This IS the baseline
                    baselines_per_image[img_name] = metrics["frames"]
                    quality = {"ssim_mean": 1.0, "psnr_mean": float("inf"), "lpips_mean": 0.0}
                else:
                    ref_frames = baselines_per_image.get(img_name)
                    if ref_frames:
                        quality = compute_quality_metrics(metrics["frames"], ref_frames)
                    else:
                        quality = {"ssim_mean": None, "psnr_mean": None, "lpips_mean": None}

                per_image_metrics.append(metrics)
                per_image_quality.append(quality)

            except Exception as e:
                print(f"   ⚠️ Failed on '{img_name}': {e}")

        if not per_image_metrics:
            print(f"   ❌ All images failed for config: {config_name}")
            all_results.append({
                "config": config_name, "time_s": None, "peak_vram_gb": None,
                "energy_wh": None, "ssim": None, "psnr_db": None, "lpips": None,
                "error": "all images failed",
            })
            continue

        # Average metrics across images
        avg_time = np.mean([m["gen_time_s"] for m in per_image_metrics])
        avg_vram = np.mean([m["peak_vram_gb"] for m in per_image_metrics])
        avg_energy = np.mean([m["energy_wh"] for m in per_image_metrics])
        avg_ssim = np.mean([q["ssim_mean"] for q in per_image_quality if q["ssim_mean"] is not None])
        avg_psnr = np.mean([q["psnr_mean"] for q in per_image_quality
                           if q["psnr_mean"] is not None and q["psnr_mean"] != float("inf")])
        avg_lpips_vals = [q.get("lpips_mean") for q in per_image_quality if q.get("lpips_mean") is not None]
        avg_lpips = np.mean(avg_lpips_vals) if avg_lpips_vals else None

        # Print summary
        if config_name == list(configs.keys())[0]:
            print(f"   ✅ BASELINE: {avg_time:.1f}s | {avg_vram:.2f} GB | {avg_energy:.3f} Wh"
                  + (f" (avg of {n_images} images)" if is_multi else ""))
        else:
            print(f"   ✅ {avg_time:.1f}s | {avg_vram:.2f} GB | SSIM={avg_ssim:.4f} | PSNR={avg_psnr:.1f}dB"
                  + (f" | LPIPS={avg_lpips:.4f}" if avg_lpips is not None else "")
                  + (f" (avg of {n_images})" if is_multi else ""))

        result_entry = {
            "config": config_name,
            "time_s": round(avg_time, 2),
            "peak_vram_gb": round(avg_vram, 3),
            "energy_wh": round(avg_energy, 4),
            "effective_steps": per_image_metrics[0]["effective_steps"],
            "num_frames": per_image_metrics[0]["num_frames"],
            "ssim": round(float(avg_ssim), 4) if not np.isnan(avg_ssim) else 1.0,
            "psnr_db": round(float(avg_psnr), 2) if (not np.isnan(avg_psnr) and avg_psnr != float("inf")) else 99.0,
            "lpips": round(float(avg_lpips), 4) if avg_lpips is not None else None,
            "n_images": n_images,
        }

        # Per-image detail (for dataset mode analysis)
        if is_multi:
            result_entry["per_image_times"] = [m["gen_time_s"] for m in per_image_metrics]
            result_entry["per_image_vram"] = [m["peak_vram_gb"] for m in per_image_metrics]
            result_entry["per_image_ssim"] = [q["ssim_mean"] for q in per_image_quality if q["ssim_mean"] is not None]

        all_results.append(result_entry)

    return all_results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RUN BENCHMARKS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 70)
print("🔬 OPTIMIZATION BENCHMARK — Starting")
print(f"   Mode: {IMAGE_MODE} | Images: {len(benchmark_images)}")
print(f"   Preset: {PRESET_NAME} | Frames: {NUM_FRAMES} | Seed: {SEED}")
print(f"   GPU: {torch.cuda.get_device_name()}")
print(f"   Configs to test: {len(BENCHMARK_CONFIGS)}")
print("=" * 70)

# Run optimization configs
results = run_benchmark_across_images(benchmark_images, BENCHMARK_CONFIGS, PRESET_NAME, NUM_FRAMES)

# Run preset comparison
print("\n" + "=" * 70)
print("🔬 PRESET COMPARISON (with default optimizations)")
print("=" * 70)

for config_name, (preset_name, opts) in PRESET_COMPARISON.items():
    print(f"\n   Running: {config_name}...")
    try:
        # Use first image for preset comparison
        metrics = run_single_benchmark(benchmark_images[0]["image"], opts, preset_name)
        # Compare vs eco baseline
        results.append({
            "config": f"[PRESET] {config_name}",
            "time_s": metrics["gen_time_s"],
            "peak_vram_gb": metrics["peak_vram_gb"],
            "energy_wh": metrics["energy_wh"],
            "effective_steps": metrics["effective_steps"],
            "num_frames": metrics["num_frames"],
            "ssim": None,  # Different frame counts, can't compare
            "psnr_db": None,
            "lpips": None,
            "n_images": 1,
        })
        print(f"   ✅ {metrics['gen_time_s']:.1f}s | {metrics['peak_vram_gb']:.2f} GB | {metrics['energy_wh']:.3f} Wh")
    except Exception as e:
        print(f"   ❌ FAILED: {e}")

print("\n✅ Benchmark complete!")
print(f"   Total results: {len(results)}")

In [ ]:
"""Cell 7: Visualize Benchmark Results"""

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BUILD DATAFRAME & SAVE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

df = pd.DataFrame(results)
df = df.dropna(subset=["time_s"])  # Remove failed runs

# Compute relative metrics (vs baseline)
baseline = df.iloc[0]
df["time_vs_baseline_%"] = ((df["time_s"] - baseline["time_s"]) / baseline["time_s"] * 100).round(1)
df["vram_vs_baseline_%"] = ((df["peak_vram_gb"] - baseline["peak_vram_gb"]) / baseline["peak_vram_gb"] * 100).round(1)
df["energy_vs_baseline_%"] = ((df["energy_wh"] - baseline["energy_wh"]) / baseline["energy_wh"] * 100).round(1)

# Save to CSV
csv_path = Path("/kaggle/working/optimization_benchmark.csv")
df.to_csv(csv_path, index=False)
print(f"📁 Results saved: {csv_path}")

# Display table
print("\n" + "=" * 90)
print("📊 BENCHMARK RESULTS TABLE")
print("=" * 90)
display_cols = ["config", "time_s", "peak_vram_gb", "energy_wh", "ssim", "psnr_db", "lpips",
                "time_vs_baseline_%", "vram_vs_baseline_%"]
print(df[display_cols].to_string(index=False))

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# VISUALIZATION
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

sns.set_theme(style="whitegrid", palette="viridis")
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle("🌿 EcoVideo Optimization Benchmark", fontsize=16, fontweight="bold")

# Filter to just optimization configs (not preset comparison)
df_opt = df[~df["config"].str.startswith("[PRESET]")].copy()
df_preset = df[df["config"].str.startswith("[PRESET]")].copy()

# 1. Time comparison
ax = axes[0, 0]
bars = ax.barh(df_opt["config"], df_opt["time_s"], color=sns.color_palette("viridis", len(df_opt)))
ax.set_xlabel("Generation Time (seconds)")
ax.set_title("⏱️ Time per Configuration")
ax.axvline(x=baseline["time_s"], color="red", linestyle="--", alpha=0.7, label="baseline")
ax.legend()
for bar, val in zip(bars, df_opt["time_vs_baseline_%"]):
    sign = "+" if val > 0 else ""
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{sign}{val}%", va="center", fontsize=8, color="gray")

# 2. Peak VRAM
ax = axes[0, 1]
colors = ["#e74c3c" if v > baseline["peak_vram_gb"] else "#2ecc71" for v in df_opt["peak_vram_gb"]]
bars = ax.barh(df_opt["config"], df_opt["peak_vram_gb"], color=colors)
ax.set_xlabel("Peak VRAM (GB)")
ax.set_title("💾 Peak VRAM Usage")
ax.axvline(x=baseline["peak_vram_gb"], color="red", linestyle="--", alpha=0.7)

# 3. Energy
ax = axes[0, 2]
ax.barh(df_opt["config"], df_opt["energy_wh"], color=sns.color_palette("YlOrRd", len(df_opt)))
ax.set_xlabel("Energy (Wh)")
ax.set_title("⚡ Energy Consumption")
ax.axvline(x=baseline["energy_wh"], color="red", linestyle="--", alpha=0.7)

# 4. SSIM (quality)
ax = axes[1, 0]
ssim_colors = ["#2ecc71" if s >= 0.95 else "#f39c12" if s >= 0.85 else "#e74c3c" for s in df_opt["ssim"]]
ax.barh(df_opt["config"], df_opt["ssim"], color=ssim_colors)
ax.set_xlabel("SSIM (1.0 = identical to baseline)")
ax.set_title("🎨 Structural Similarity (SSIM)")
ax.set_xlim(0, 1.05)
ax.axvline(x=0.95, color="green", linestyle=":", alpha=0.5, label="excellent (≥0.95)")
ax.axvline(x=0.85, color="orange", linestyle=":", alpha=0.5, label="acceptable (≥0.85)")
ax.legend(fontsize=8)

# 5. PSNR
ax = axes[1, 1]
psnr_vals = df_opt["psnr_db"].clip(upper=50)  # Cap for display
ax.barh(df_opt["config"], psnr_vals, color=sns.color_palette("Blues_r", len(df_opt)))
ax.set_xlabel("PSNR (dB) — higher is better")
ax.set_title("📐 Peak Signal-to-Noise Ratio")
ax.axvline(x=30, color="green", linestyle=":", alpha=0.5, label="good (≥30dB)")
ax.axvline(x=20, color="orange", linestyle=":", alpha=0.5, label="acceptable (≥20dB)")
ax.legend(fontsize=8)

# 6. Trade-off scatter: Time vs Quality
ax = axes[1, 2]
scatter = ax.scatter(
    df_opt["time_s"], df_opt["ssim"],
    s=df_opt["peak_vram_gb"] * 40,  # Size = VRAM
    c=df_opt["energy_wh"],
    cmap="RdYlGn_r",
    alpha=0.8,
    edgecolors="black",
    linewidth=0.5,
)
for _, row in df_opt.iterrows():
    ax.annotate(row["config"].split(" ")[0], (row["time_s"], row["ssim"]),
                fontsize=6, alpha=0.7, ha="center", va="bottom")
ax.set_xlabel("Time (s)")
ax.set_ylabel("SSIM (quality)")
ax.set_title("🎯 Time vs Quality Trade-off\n(size=VRAM, color=energy)")
plt.colorbar(scatter, ax=ax, label="Energy (Wh)")

plt.tight_layout()
plt.savefig("/kaggle/working/optimization_benchmark_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Chart saved: /kaggle/working/optimization_benchmark_chart.png")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PRESET COMPARISON CHART
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
if len(df_preset) > 0:
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle("🌿 Preset Comparison (eco vs balanced vs quality)", fontsize=14)

    preset_colors = ["#4CAF50", "#FF9800", "#2196F3"]

    for ax, (col, title, unit) in zip(axes, [
        ("time_s", "Time", "s"), ("peak_vram_gb", "Peak VRAM", "GB"),
        ("energy_wh", "Energy", "Wh"), ("ssim", "Quality (SSIM)", "")
    ]):
        ax.bar(df_preset["config"].str.replace("[PRESET] ", ""), df_preset[col], color=preset_colors)
        ax.set_title(title)
        ax.set_ylabel(f"{title} ({unit})" if unit else title)
        ax.tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.savefig("/kaggle/working/preset_comparison_chart.png", dpi=150, bbox_inches="tight")
    plt.show()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SUMMARY TABLE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n" + "=" * 90)
print("🏆 TOP FINDINGS")
print("=" * 90)

if len(df_opt) > 1:
    fastest = df_opt.loc[df_opt["time_s"].idxmin()]
    least_vram = df_opt.loc[df_opt["peak_vram_gb"].idxmin()]
    least_energy = df_opt.loc[df_opt["energy_wh"].idxmin()]
    best_quality = df_opt.loc[df_opt["ssim"].idxmax()]
    
    # Best trade-off: highest SSIM with time < baseline
    efficient = df_opt[df_opt["time_s"] <= baseline["time_s"] * 1.05]
    if len(efficient) > 1:
        best_tradeoff = efficient.loc[efficient["ssim"].idxmax()]
    else:
        best_tradeoff = fastest

    print(f"\n  ⚡ Fastest:       {fastest['config']} — {fastest['time_s']:.1f}s ({fastest['time_vs_baseline_%']:+.0f}% vs baseline)")
    print(f"  💾 Least VRAM:    {least_vram['config']} — {least_vram['peak_vram_gb']:.2f} GB ({least_vram['vram_vs_baseline_%']:+.0f}%)")
    print(f"  🔋 Least Energy:  {least_energy['config']} — {least_energy['energy_wh']:.3f} Wh ({least_energy['energy_vs_baseline_%']:+.0f}%)")
    print(f"  🎨 Best Quality:  {best_quality['config']} — SSIM={best_quality['ssim']:.4f}")
    print(f"  🎯 Best Trade-off:{best_tradeoff['config']} — {best_tradeoff['time_s']:.1f}s, SSIM={best_tradeoff['ssim']:.4f}")

print("\n✅ Full results in /kaggle/working/optimization_benchmark.csv")

# 🧪 Dataset-Based Optimization Testing & Report Generation

This section loads a **small open-source image dataset** from HuggingFace, runs all optimization configurations and presets through the SVD pipeline, and generates comprehensive visual charts for the research report.

### Dataset Options (choose one):
| Dataset | Total Images | Download Size | Description |
|---------|-------------|---------------|-------------|
| `huggan/flowers-102-categories` | **8,189 images** | 347 MB | Oxford Flowers — diverse natural scenes |
| `lambdalabs/naruto-blip-captions` | **~1,000 images** | 701 MB | Anime-style with BLIP captions |
| `CIFAR-10` (torchvision) | **60,000 images** | 170 MB | Classic 32×32 (upscaled to 1024×576) |

### ⏱️ Time Estimates (T4 GPU, 14 configs tested):
| Images | Est. Total Time | Use Case |
|--------|----------------|----------|
| **1** | ~8 min | Quick smoke test |
| **3** | ~25 min | ✅ Good for report (recommended) |
| **5** | ~40 min | More statistically robust |
| **10** | ~80 min | Comprehensive (if time allows) |

> ⚠️ **Formula**: Total time ≈ 14 configs × N images × ~35s average per run  
> 💡 Set `NUM_TEST_IMAGES` in the next cell. **Start with 3**, increase if you have time.

In [ ]:
"""Cell 8: Install dataset dependencies & load HuggingFace dataset"""

!pip install -q datasets pillow

import gc
import time
import random
from pathlib import Path

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# DATASET SELECTION — Pick ONE by uncommenting
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ╔══════════════════════════════════════════════════════════════╗
# ║  👇 CONFIGURE THESE TWO SETTINGS BEFORE RUNNING            ║
# ╚══════════════════════════════════════════════════════════════╝

DATASET_CHOICE = "flowers"  # Options: "flowers", "naruto", "cifar10"

# 🎯 NUMBER OF IMAGES TO TEST (controls total runtime)
# ┌─────────────────────────────────────────────────────────┐
# │  1 image  → ~8 min  (quick smoke test)                  │
# │  3 images → ~25 min (good for report, recommended)      │
# │  5 images → ~40 min (more statistically robust)         │
# │  10 images → ~80 min (comprehensive, if time allows)    │
# └─────────────────────────────────────────────────────────┘
NUM_TEST_IMAGES = 3  # ← CHANGE THIS (start small!)

def load_test_dataset(choice: str, n: int = 5):
    """Load a small sample from the chosen dataset."""
    
    if choice == "flowers":
        from datasets import load_dataset
        print("📥 Loading huggan/flowers-102-categories from HuggingFace...")
        ds = load_dataset("huggan/flowers-102-categories", split="train")
        indices = random.sample(range(len(ds)), min(n, len(ds)))
        images = []
        for i, idx in enumerate(indices):
            img = ds[idx]["image"].convert("RGB").resize((1024, 576))
            images.append({"name": f"flower_{i+1}", "image": img})
        print(f"✅ Loaded {len(images)} flower images (resized to 1024×576)")
        
    elif choice == "naruto":
        from datasets import load_dataset
        print("📥 Loading lambda/naruto-blip-captions from HuggingFace...")
        ds = load_dataset("lambdalabs/naruto-blip-captions", split="train")
        indices = random.sample(range(len(ds)), min(n, len(ds)))
        images = []
        for i, idx in enumerate(indices):
            img = ds[idx]["image"].convert("RGB").resize((1024, 576))
            images.append({"name": f"naruto_{i+1}", "image": img})
        print(f"✅ Loaded {len(images)} naruto images (resized to 1024×576)")
        
    elif choice == "cifar10":
        import torchvision
        print("📥 Loading CIFAR-10 (built-in torchvision)...")
        cifar = torchvision.datasets.CIFAR10(root="/kaggle/working/cifar10", 
                                              train=True, download=True)
        indices = random.sample(range(len(cifar)), min(n, len(cifar)))
        images = []
        for i, idx in enumerate(indices):
            img = cifar[idx][0].convert("RGB").resize((1024, 576), Image.LANCZOS)
            images.append({"name": f"cifar_{i+1}", "image": img})
        print(f"✅ Loaded {len(images)} CIFAR-10 images (upscaled to 1024×576)")
    
    else:
        raise ValueError(f"Unknown dataset choice: {choice}")
    
    return images

# Set seed for reproducibility
random.seed(42)
torch.manual_seed(42)

# Load dataset
test_images = load_test_dataset(DATASET_CHOICE, NUM_TEST_IMAGES)

# Show preview
fig, axes = plt.subplots(1, len(test_images), figsize=(4*len(test_images), 4))
if len(test_images) == 1:
    axes = [axes]
for ax, img_info in zip(axes, test_images):
    ax.imshow(img_info["image"])
    ax.set_title(img_info["name"], fontsize=10)
    ax.axis("off")
plt.suptitle(f"Test Dataset: {DATASET_CHOICE} ({len(test_images)} images)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/test_dataset_preview.png", dpi=100)
plt.show()
print(f"\n📊 Dataset ready: {len(test_images)} images for benchmarking")

In [ ]:
"""Cell 9: Run Full Optimization Test Matrix on Dataset"""

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FULL TEST MATRIX: All Presets × All Optimizations × Dataset Images
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Test configurations matching our report
TEST_MATRIX = {
    # ─── Individual Optimizations ───────────────────────────────
    "Baseline (no opts)": {
        "preset": "balanced",
        "attention_slicing": False,
        "forward_chunking": False,
        "deep_cache": False,
        "cpu_offload": False,
    },
    "Attention Slicing": {
        "preset": "balanced",
        "attention_slicing": True,
        "forward_chunking": False,
        "deep_cache": False,
        "cpu_offload": False,
    },
    "Forward Chunking": {
        "preset": "balanced",
        "attention_slicing": False,
        "forward_chunking": True,
        "deep_cache": False,
        "cpu_offload": False,
    },
    "DeepCache (interval=2)": {
        "preset": "balanced",
        "attention_slicing": False,
        "forward_chunking": False,
        "deep_cache": True,
        "deep_cache_interval": 2,
        "cpu_offload": False,
    },
    "DeepCache (interval=3)": {
        "preset": "balanced",
        "attention_slicing": False,
        "forward_chunking": False,
        "deep_cache": True,
        "deep_cache_interval": 3,
        "cpu_offload": False,
    },
    "DeepCache (interval=5)": {
        "preset": "balanced",
        "attention_slicing": False,
        "forward_chunking": False,
        "deep_cache": True,
        "deep_cache_interval": 5,
        "cpu_offload": False,
    },
    "CPU Offload": {
        "preset": "balanced",
        "attention_slicing": False,
        "forward_chunking": False,
        "deep_cache": False,
        "cpu_offload": True,
    },
    # ─── Combinations ──────────────────────────────────────────
    "AttnSlice + FwdChunk": {
        "preset": "balanced",
        "attention_slicing": True,
        "forward_chunking": True,
        "deep_cache": False,
        "cpu_offload": False,
    },
    "AttnSlice + DeepCache(3)": {
        "preset": "balanced",
        "attention_slicing": True,
        "forward_chunking": False,
        "deep_cache": True,
        "deep_cache_interval": 3,
        "cpu_offload": False,
    },
    "All Opts (no offload)": {
        "preset": "balanced",
        "attention_slicing": True,
        "forward_chunking": True,
        "deep_cache": True,
        "deep_cache_interval": 3,
        "cpu_offload": False,
    },
    "All Opts + CPU Offload": {
        "preset": "balanced",
        "attention_slicing": True,
        "forward_chunking": True,
        "deep_cache": True,
        "deep_cache_interval": 3,
        "cpu_offload": True,
    },
    # ─── Preset Variations (with best combo: attn+fwd+cache3) ──
    "ECO preset (8 steps)": {
        "preset": "eco",
        "attention_slicing": True,
        "forward_chunking": True,
        "deep_cache": True,
        "deep_cache_interval": 3,
        "cpu_offload": False,
    },
    "BALANCED preset (16 steps)": {
        "preset": "balanced",
        "attention_slicing": True,
        "forward_chunking": True,
        "deep_cache": True,
        "deep_cache_interval": 3,
        "cpu_offload": False,
    },
    "QUALITY preset (25 steps)": {
        "preset": "quality",
        "attention_slicing": True,
        "forward_chunking": True,
        "deep_cache": True,
        "deep_cache_interval": 3,
        "cpu_offload": False,
    },
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BENCHMARK RUNNER
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

GPU_TDP_WATTS = 70.0  # T4 TDP
GPU_UTILIZATION_FACTOR = 0.8
SEED = 42

all_results = []
baseline_frames_per_image = {}

print("=" * 80)
print("🔬 FULL OPTIMIZATION TEST — Dataset-Based Benchmark")
print(f"   Dataset: {DATASET_CHOICE} | Images: {len(test_images)}")
print(f"   Configurations: {len(TEST_MATRIX)}")
print(f"   GPU: {torch.cuda.get_device_name()}")
print("=" * 80)

for config_idx, (config_name, config) in enumerate(TEST_MATRIX.items()):
    print(f"\n{'─'*60}")
    print(f"[{config_idx+1}/{len(TEST_MATRIX)}] {config_name}")
    print(f"{'─'*60}")
    
    preset_name = config["preset"]
    preset = PRESETS[preset_name]
    num_frames = min(6, preset["max_frames"])
    
    # Determine effective steps with DeepCache
    use_deep_cache = config.get("deep_cache", False)
    cache_interval = config.get("deep_cache_interval", 3)
    if use_deep_cache:
        effective_steps = max(preset["steps"] // cache_interval * 2, 4)
    else:
        effective_steps = preset["steps"]
    
    # Apply optimizations
    opts = {k: v for k, v in config.items() if k != "preset"}
    _apply_optimizations(pipe, opts)
    
    img_times = []
    img_vrams = []
    img_energies = []
    img_ssims = []
    img_psnrs = []
    
    for img_info in test_images:
        image = img_info["image"]
        img_name = img_info["name"]
        
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
        generator = torch.Generator(device="cuda").manual_seed(SEED)
        
        try:
            start = time.time()
            with torch.no_grad():
                output = pipe(
                    image=image,
                    decode_chunk_size=preset["decode_chunk"],
                    generator=generator,
                    num_frames=num_frames,
                    motion_bucket_id=127,
                    noise_aug_strength=0.02,
                    num_inference_steps=effective_steps,
                )
            gen_time = time.time() - start
            
            frames = output.frames[0]
            peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
            energy_wh = (GPU_TDP_WATTS * GPU_UTILIZATION_FACTOR * gen_time) / 3600.0
            
            img_times.append(gen_time)
            img_vrams.append(peak_vram)
            img_energies.append(energy_wh)
            
            # Quality comparison vs baseline
            if config_name == "Baseline (no opts)":
                baseline_frames_per_image[img_name] = frames
                img_ssims.append(1.0)
                img_psnrs.append(99.0)
            else:
                ref = baseline_frames_per_image.get(img_name)
                if ref and len(ref) > 0 and len(frames) > 0:
                    n_compare = min(len(frames), len(ref))
                    ssim_vals, psnr_vals = [], []
                    for fi in range(n_compare):
                        a = np.array(frames[fi].resize((256, 144)))
                        b = np.array(ref[fi].resize((256, 144)))
                        ssim_vals.append(ssim(a, b, channel_axis=2, data_range=255))
                        psnr_vals.append(psnr(b, a, data_range=255))
                    img_ssims.append(np.mean(ssim_vals))
                    img_psnrs.append(np.mean(psnr_vals))
                else:
                    img_ssims.append(None)
                    img_psnrs.append(None)
                    
            print(f"   ✅ {img_name}: {gen_time:.1f}s | {peak_vram:.2f}GB | {energy_wh:.4f}Wh")
            
        except Exception as e:
            print(f"   ❌ {img_name}: FAILED — {e}")
    
    # Aggregate across images
    if img_times:
        valid_ssims = [s for s in img_ssims if s is not None]
        valid_psnrs = [p for p in img_psnrs if p is not None]
        
        result = {
            "config": config_name,
            "preset": preset_name,
            "effective_steps": effective_steps,
            "num_frames": num_frames,
            "n_images": len(img_times),
            "avg_time_s": round(np.mean(img_times), 2),
            "std_time_s": round(np.std(img_times), 2),
            "avg_vram_gb": round(np.mean(img_vrams), 3),
            "avg_energy_wh": round(np.mean(img_energies), 4),
            "total_energy_wh": round(np.sum(img_energies), 4),
            "avg_ssim": round(np.mean(valid_ssims), 4) if valid_ssims else None,
            "avg_psnr_db": round(np.mean(valid_psnrs), 2) if valid_psnrs else None,
            "carbon_g_per_video": round(np.mean(img_energies) / 1000 * 475, 6),  # gCO2 @ 475 gCO2/kWh
        }
        all_results.append(result)
        
        print(f"   📊 AVG: {result['avg_time_s']:.1f}s ± {result['std_time_s']:.1f} | "
              f"VRAM: {result['avg_vram_gb']:.2f}GB | Energy: {result['avg_energy_wh']:.4f}Wh | "
              f"SSIM: {result['avg_ssim']}" + 
              (f" | PSNR: {result['avg_psnr_db']}dB" if result['avg_psnr_db'] else ""))

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SAVE RESULTS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

df_results = pd.DataFrame(all_results)

# Add derived columns
baseline_row = df_results[df_results["config"] == "Baseline (no opts)"].iloc[0]
df_results["speedup_vs_baseline"] = round(baseline_row["avg_time_s"] / df_results["avg_time_s"], 2)
df_results["vram_reduction_%"] = round((1 - df_results["avg_vram_gb"] / baseline_row["avg_vram_gb"]) * 100, 1)
df_results["energy_savings_%"] = round((1 - df_results["avg_energy_wh"] / baseline_row["avg_energy_wh"]) * 100, 1)

# Cloud comparison (A100 baseline: ~150 Wh per video, 475 gCO2/kWh)
CLOUD_ENERGY_WH = 150.0
CLOUD_CARBON_G = CLOUD_ENERGY_WH / 1000 * 475
df_results["vs_cloud_energy_reduction_%"] = round((1 - df_results["avg_energy_wh"] / CLOUD_ENERGY_WH) * 100, 2)
df_results["vs_cloud_carbon_reduction_%"] = round((1 - df_results["carbon_g_per_video"] / CLOUD_CARBON_G) * 100, 2)

# Save
csv_path = Path("/kaggle/working/dataset_benchmark_full.csv")
df_results.to_csv(csv_path, index=False)
print(f"\n\n💾 Results saved: {csv_path}")
print(f"📊 Total configurations tested: {len(df_results)}")

# Display table
print("\n" + "=" * 100)
print("📋 RESULTS SUMMARY TABLE")
print("=" * 100)
display_cols = ["config", "preset", "effective_steps", "avg_time_s", "avg_vram_gb", 
                "avg_energy_wh", "avg_ssim", "speedup_vs_baseline", "energy_savings_%"]
print(df_results[display_cols].to_string(index=False))
print("=" * 100)

In [ ]:
"""Cell 10: Generate Report Visualizations — All Charts for Findings"""

sns.set_theme(style="whitegrid")
SAVE_DIR = Path("/kaggle/working/report_charts")
SAVE_DIR.mkdir(exist_ok=True)

# Separate optimization configs from preset comparisons
df_opts = df_results[~df_results["config"].str.contains("preset")].copy()
df_presets = df_results[df_results["config"].str.contains("preset")].copy()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 1: Speed Comparison — All Optimization Configs
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, ax = plt.subplots(figsize=(14, 7))
colors = sns.color_palette("viridis", len(df_opts))
bars = ax.barh(df_opts["config"], df_opts["avg_time_s"], color=colors, edgecolor="black", linewidth=0.5)
ax.axvline(x=baseline_row["avg_time_s"], color="red", linestyle="--", alpha=0.7, linewidth=2, label="Baseline")
ax.set_xlabel("Average Generation Time (seconds)", fontsize=12)
ax.set_title("⏱️ Generation Speed: All Optimization Configurations\n"
             f"(Dataset: {DATASET_CHOICE}, {len(test_images)} images averaged)", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
# Add speedup labels
for bar, speedup in zip(bars, df_opts["speedup_vs_baseline"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{speedup:.1f}×", va="center", fontsize=9, color="darkblue", fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "01_speed_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 2: VRAM Usage Comparison
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, ax = plt.subplots(figsize=(14, 7))
vram_colors = ["#e74c3c" if v >= baseline_row["avg_vram_gb"] else "#2ecc71" 
               for v in df_opts["avg_vram_gb"]]
bars = ax.barh(df_opts["config"], df_opts["avg_vram_gb"], color=vram_colors, edgecolor="black", linewidth=0.5)
ax.axvline(x=baseline_row["avg_vram_gb"], color="red", linestyle="--", alpha=0.7, linewidth=2, label="Baseline")
ax.set_xlabel("Peak VRAM (GB)", fontsize=12)
ax.set_title("💾 Peak VRAM Usage: Memory Optimization Effect\n"
             f"(Green = reduced vs baseline, Red = same/higher)", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
for bar, pct in zip(bars, df_opts["vram_reduction_%"]):
    label = f"{pct:+.0f}%" if pct != 0 else "baseline"
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            label, va="center", fontsize=9)
plt.tight_layout()
plt.savefig(SAVE_DIR / "02_vram_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 3: Energy & Carbon Emissions (Dual Axis)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, ax1 = plt.subplots(figsize=(14, 7))
x = np.arange(len(df_opts))
width = 0.4

bars1 = ax1.bar(x - width/2, df_opts["avg_energy_wh"], width, label="Energy (Wh)",
                color="#f39c12", edgecolor="black", linewidth=0.5)
ax1.set_ylabel("Energy per Video (Wh)", color="#f39c12", fontsize=12)
ax1.tick_params(axis="y", labelcolor="#f39c12")

ax2 = ax1.twinx()
bars2 = ax2.bar(x + width/2, df_opts["carbon_g_per_video"] * 1000, width, label="Carbon (mgCO₂)",
                color="#27ae60", edgecolor="black", linewidth=0.5)
ax2.set_ylabel("Carbon Emissions (mgCO₂/video)", color="#27ae60", fontsize=12)
ax2.tick_params(axis="y", labelcolor="#27ae60")

ax1.set_xticks(x)
ax1.set_xticklabels(df_opts["config"], rotation=45, ha="right", fontsize=9)
ax1.set_title("⚡ Energy Consumption & 🌱 Carbon Footprint per Video\n"
              f"(475 gCO₂/kWh global average)", fontsize=13, fontweight="bold")
fig.legend(loc="upper right", bbox_to_anchor=(0.95, 0.95), fontsize=11)
plt.tight_layout()
plt.savefig(SAVE_DIR / "03_energy_carbon.png", dpi=150, bbox_inches="tight")
plt.show()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 4: Quality vs Speed Trade-off (Scatter)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, ax = plt.subplots(figsize=(12, 8))
df_quality = df_opts[df_opts["avg_ssim"].notna()].copy()

scatter = ax.scatter(
    df_quality["avg_time_s"],
    df_quality["avg_ssim"],
    s=df_quality["avg_vram_gb"] * 50,
    c=df_quality["avg_energy_wh"],
    cmap="RdYlGn_r",
    alpha=0.85,
    edgecolors="black",
    linewidth=0.8,
    zorder=3,
)

# Annotate points
for _, row in df_quality.iterrows():
    short_name = row["config"].replace(" (no opts)", "").replace(" only", "")
    ax.annotate(short_name, (row["avg_time_s"], row["avg_ssim"]),
                fontsize=7, alpha=0.8, ha="center", va="bottom",
                xytext=(0, 8), textcoords="offset points")

ax.set_xlabel("Generation Time (seconds) →", fontsize=12)
ax.set_ylabel("← Quality (SSIM vs Baseline)", fontsize=12)
ax.set_title("🎯 Quality vs Speed Trade-off\n"
             "(bubble size = VRAM, color = energy)", fontsize=13, fontweight="bold")
plt.colorbar(scatter, ax=ax, label="Energy (Wh)", shrink=0.8)
ax.axhline(y=0.95, color="green", linestyle=":", alpha=0.5, label="Excellent quality (≥0.95)")
ax.axhline(y=0.85, color="orange", linestyle=":", alpha=0.5, label="Acceptable quality (≥0.85)")
ax.legend(loc="lower right", fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_DIR / "04_quality_vs_speed.png", dpi=150, bbox_inches="tight")
plt.show()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 5: Preset Comparison (Eco vs Balanced vs Quality)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
if len(df_presets) >= 3:
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    fig.suptitle("🌿 Preset Comparison: ECO vs BALANCED vs QUALITY\n"
                 "(all with AttnSlice + FwdChunk + DeepCache(3))", fontsize=13, fontweight="bold")
    
    preset_names = df_presets["config"].str.extract(r"(ECO|BALANCED|QUALITY)")[0].tolist()
    preset_colors = {"ECO": "#4CAF50", "BALANCED": "#FF9800", "QUALITY": "#2196F3"}
    colors = [preset_colors.get(p, "#999") for p in preset_names]
    
    metrics = [
        ("avg_time_s", "Time (s)", "⏱️ Generation Time"),
        ("avg_vram_gb", "Peak VRAM (GB)", "💾 VRAM Usage"),
        ("avg_energy_wh", "Energy (Wh)", "⚡ Energy"),
        ("avg_ssim", "SSIM", "🎨 Quality (SSIM)"),
    ]
    
    for ax, (col, ylabel, title) in zip(axes, metrics):
        vals = df_presets[col].tolist()
        ax.bar(preset_names, vals, color=colors, edgecolor="black", linewidth=0.5)
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontsize=11)
        for i, v in enumerate(vals):
            if v is not None:
                ax.text(i, v, f"{v:.3f}" if v < 1 else f"{v:.1f}", 
                       ha="center", va="bottom", fontsize=9)
    
    plt.tight_layout()
    plt.savefig(SAVE_DIR / "05_preset_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 6: Cloud vs Local — Sustainability Impact
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("☁️ Cloud A100 vs 🖥️ Local T4 (Kaggle) — Sustainability Comparison\n"
             f"Dataset: {DATASET_CHOICE} ({len(test_images)} images)", fontsize=13, fontweight="bold")

# Best local config
best_local = df_results.loc[df_results["energy_savings_%"].idxmax()]
balanced_opt = df_results[df_results["config"] == "All Opts (no offload)"]
if len(balanced_opt) > 0:
    best_balanced = balanced_opt.iloc[0]
else:
    best_balanced = best_local

labels = ["Cloud A100\n(estimated)", f"Local T4\nBaseline", f"Local T4\nOptimized"]
bar_colors = ["#e74c3c", "#3498db", "#2ecc71"]

# Energy
energy_vals = [CLOUD_ENERGY_WH, baseline_row["avg_energy_wh"], best_balanced["avg_energy_wh"]]
axes[0].bar(labels, energy_vals, color=bar_colors, edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("Energy (Wh)")
axes[0].set_title("⚡ Energy per Video")
for i, v in enumerate(energy_vals):
    axes[0].text(i, v, f"{v:.3f}" if v < 1 else f"{v:.1f}", ha="center", va="bottom", fontsize=10)

# Carbon
carbon_vals = [CLOUD_CARBON_G, baseline_row["carbon_g_per_video"], best_balanced["carbon_g_per_video"]]
axes[1].bar(labels, carbon_vals, color=bar_colors, edgecolor="black", linewidth=0.5)
axes[1].set_ylabel("Carbon (gCO₂)")
axes[1].set_title("🌱 Carbon Emissions")
for i, v in enumerate(carbon_vals):
    axes[1].text(i, v, f"{v:.4f}" if v < 0.1 else f"{v:.2f}", ha="center", va="bottom", fontsize=10)

# Reduction percentage
reductions = [0, 
              (1 - baseline_row["avg_energy_wh"]/CLOUD_ENERGY_WH)*100,
              (1 - best_balanced["avg_energy_wh"]/CLOUD_ENERGY_WH)*100]
axes[2].bar(labels, reductions, color=bar_colors, edgecolor="black", linewidth=0.5)
axes[2].set_ylabel("Reduction vs Cloud (%)")
axes[2].set_title("📉 Energy Reduction vs Cloud")
for i, v in enumerate(reductions):
    axes[2].text(i, v + 1, f"{v:.1f}%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig(SAVE_DIR / "06_cloud_vs_local.png", dpi=150, bbox_inches="tight")
plt.show()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 7: DeepCache Interval Analysis
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
df_cache = df_results[df_results["config"].str.contains("DeepCache")].copy()
if len(df_cache) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("🔄 DeepCache Interval Analysis\n"
                 "(How cache frequency affects speed, energy, and quality)", fontsize=13, fontweight="bold")
    
    cache_labels = df_cache["config"].tolist()
    
    axes[0].bar(cache_labels, df_cache["avg_time_s"], color=["#3498db", "#2ecc71", "#9b59b6"],
                edgecolor="black", linewidth=0.5)
    axes[0].set_ylabel("Time (s)")
    axes[0].set_title("⏱️ Speed")
    axes[0].axhline(y=baseline_row["avg_time_s"], color="red", linestyle="--", label="No cache")
    axes[0].legend()
    
    axes[1].bar(cache_labels, df_cache["avg_energy_wh"], color=["#3498db", "#2ecc71", "#9b59b6"],
                edgecolor="black", linewidth=0.5)
    axes[1].set_ylabel("Energy (Wh)")
    axes[1].set_title("⚡ Energy")
    axes[1].axhline(y=baseline_row["avg_energy_wh"], color="red", linestyle="--", label="No cache")
    axes[1].legend()
    
    ssim_vals = df_cache["avg_ssim"].tolist()
    axes[2].bar(cache_labels, ssim_vals, color=["#3498db", "#2ecc71", "#9b59b6"],
                edgecolor="black", linewidth=0.5)
    axes[2].set_ylabel("SSIM")
    axes[2].set_title("🎨 Quality (SSIM)")
    axes[2].axhline(y=0.95, color="green", linestyle=":", label="Excellent")
    axes[2].axhline(y=0.85, color="orange", linestyle=":", label="Acceptable")
    axes[2].legend()
    axes[2].set_ylim(0, 1.05)
    
    plt.tight_layout()
    plt.savefig(SAVE_DIR / "07_deepcache_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 8: Comprehensive Heatmap — All Metrics
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, ax = plt.subplots(figsize=(10, 8))
heatmap_cols = ["speedup_vs_baseline", "vram_reduction_%", "energy_savings_%", "avg_ssim"]
heatmap_data = df_opts[heatmap_cols].copy()
heatmap_data.index = df_opts["config"]
heatmap_data.columns = ["Speedup (×)", "VRAM Saved (%)", "Energy Saved (%)", "Quality (SSIM)"]

sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            linewidths=0.5, ax=ax, cbar_kws={"label": "Score"})
ax.set_title("📊 Optimization Performance Heatmap\n"
             "(Green = better, Red = worse)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "08_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PRINT KEY FINDINGS FOR REPORT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n" + "=" * 80)
print("🏆 KEY FINDINGS FOR REPORT")
print("=" * 80)

fastest = df_opts.loc[df_opts["avg_time_s"].idxmin()]
least_vram = df_opts.loc[df_opts["avg_vram_gb"].idxmin()]
least_energy = df_opts.loc[df_opts["avg_energy_wh"].idxmin()]
best_quality = df_opts[df_opts["avg_ssim"].notna()].loc[df_opts[df_opts["avg_ssim"].notna()]["avg_ssim"].idxmax()]

print(f"""
📌 DATASET: {DATASET_CHOICE} ({len(test_images)} images, 1024×576)
📌 HARDWARE: {torch.cuda.get_device_name()} (Kaggle)

1️⃣  SPEED CHAMPION: {fastest['config']}
    → {fastest['avg_time_s']:.1f}s ({fastest['speedup_vs_baseline']:.1f}× faster than baseline)

2️⃣  MEMORY CHAMPION: {least_vram['config']}
    → {least_vram['avg_vram_gb']:.2f} GB peak ({least_vram['vram_reduction_%']:.0f}% reduction)

3️⃣  ENERGY CHAMPION: {least_energy['config']}
    → {least_energy['avg_energy_wh']:.4f} Wh/video ({least_energy['energy_savings_%']:.0f}% savings)

4️⃣  BEST QUALITY: {best_quality['config']}
    → SSIM={best_quality['avg_ssim']:.4f} (highest similarity to baseline)

5️⃣  BEST TRADE-OFF: All Opts (no offload)
    → Combines speed + memory + energy savings with acceptable quality

6️⃣  vs CLOUD (A100, 150 Wh baseline):
    → Local baseline: {(1 - baseline_row['avg_energy_wh'] / CLOUD_ENERGY_WH) * 100:.1f}% less energy than cloud A100
    → Local optimized: {(1 - best_balanced['avg_energy_wh'] / CLOUD_ENERGY_WH) * 100:.1f}% less energy than cloud A100
    → Carbon saved: {CLOUD_CARBON_G - best_balanced['carbon_g_per_video']:.4f} gCO₂ per video

7️⃣  PRESET RECOMMENDATION:
    → ECO: Maximum efficiency, acceptable for previews/drafts
    → BALANCED: Best trade-off for production use
    → QUALITY: When visual fidelity is paramount
""")

print(f"\n📁 All charts saved to: {SAVE_DIR}")
print(f"📁 CSV data: {csv_path}")
print("=" * 80)